In [1]:
%load_ext autoreload
%autoreload 2
import os
print(os.getcwd())
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
import pickle
from utils import reproducibility
batch_size = 256
reproducibility(2025)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/disk1/user/liaoshuilin/project/35.TAPE_EXO/assay_diffusion


# 数据仿真

In [2]:
%load_ext autoreload
%autoreload 2
from data_process import generate_simulated_data

simudata_GTE, label_GTE = generate_simulated_data(sc_data="../result/expr/dat1_gtex_tissue_tpm_filter.txt",
                                   n=500, samplenum=5000, 
                                   d_prior=None, sparse=True)

simudata_HPA, label_HPA = generate_simulated_data(sc_data="../result/expr/dat1_hpa_tissue_tpm_filter.txt",
                                   n=500, samplenum=50, 
                                   d_prior=None, sparse=True)

print(simudata_GTE.shape)
print(label_GTE)
print(simudata_HPA.shape)
print(label_HPA)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Reading single-cell dataset, this may take 1 min
Index(['Adipose Tissue', 'Muscle', 'Heart', 'Thyroid', 'Kidney', 'Breast',
       'Skin', 'Salivary Gland', 'Adrenal Gland', 'Thyroid',
       ...
       'Heart', 'Heart', 'Adrenal Gland', 'Thyroid', 'Stomach', 'Esophagus',
       'Ovary', 'Skin', 'Muscle', 'Adipose Tissue'],
      dtype='object', name='Tissue', length=12515)


/disk/user/liaoshuilin/software/anaconda3/envs/scGPCL/lib/python3.9/site-packages/anndata/utils.py:252: UserWarning: X converted to numpy array with dtype float64
  warnings.warn(f"{name} converted to numpy array with dtype {arr.dtype}")
/disk/user/liaoshuilin/software/anaconda3/envs/scGPCL/lib/python3.9/site-packages/anndata/_core/anndata.py:117: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Generating cell fractions using Dirichlet distribution without prior info (actually random)
You set sparse as True, some cell's fraction will be zero, the probability is 0.5
Sampling cells to compose pseudo-bulk data


5000it [00:48, 103.05it/s]
/disk/user/liaoshuilin/software/anaconda3/envs/scGPCL/lib/python3.9/site-packages/anndata/_core/anndata.py:117: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Sampling is done
Reading single-cell dataset, this may take 1 min


/disk/user/liaoshuilin/software/anaconda3/envs/scGPCL/lib/python3.9/site-packages/anndata/utils.py:252: UserWarning: X converted to numpy array with dtype float64
  warnings.warn(f"{name} converted to numpy array with dtype {arr.dtype}")
/disk/user/liaoshuilin/software/anaconda3/envs/scGPCL/lib/python3.9/site-packages/anndata/_core/anndata.py:117: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Index(['Adipose Tissue', 'Adipose Tissue', 'Adipose Tissue', 'Adipose Tissue',
       'Adipose Tissue', 'Adrenal Gland', 'Adrenal Gland', 'Adrenal Gland',
       'Bladder', 'Bladder',
       ...
       'Testis', 'Testis', 'Testis', 'Testis', 'Testis', 'Thyroid', 'Thyroid',
       'Thyroid', 'Thyroid', 'Thyroid'],
      dtype='object', name='Tissue', length=128)
Generating cell fractions using Dirichlet distribution without prior info (actually random)
You set sparse as True, some cell's fraction will be zero, the probability is 0.5
Sampling cells to compose pseudo-bulk data


50it [00:00, 138.76it/s]

Sampling is done
(5000, 18757)
dict_keys(['Adipose Tissue', 'Adrenal Gland', 'Bladder', 'Breast', 'Cervix Uteri', 'Colon', 'Esophagus', 'Fallopian Tube', 'Heart', 'Kidney', 'Liver', 'Lung', 'Muscle', 'Ovary', 'Pancreas', 'Prostate', 'Salivary Gland', 'Skin', 'Small Intestine', 'Spleen', 'Stomach', 'Testis', 'Thyroid'])
(50, 18757)
dict_keys(['Adipose Tissue', 'Adrenal Gland', 'Bladder', 'Breast', 'Cervix Uteri', 'Colon', 'Esophagus', 'Fallopian Tube', 'Heart', 'Kidney', 'Liver', 'Lung', 'Muscle', 'Ovary', 'Pancreas', 'Prostate', 'Salivary Gland', 'Skin', 'Small Intestine', 'Spleen', 'Stomach', 'Testis', 'Thyroid'])



/disk/user/liaoshuilin/software/anaconda3/envs/scGPCL/lib/python3.9/site-packages/anndata/_core/anndata.py:117: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


# 获得不同var水平的genes

In [57]:
%load_ext autoreload
%autoreload 2
from data_process import GTEDataGeneVarIntv
var_genes_intv = GTEDataGeneVarIntv(simudata_HPA)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# 不同基因数采样

In [67]:
%load_ext autoreload
%autoreload 2
from data_process import GetXandYSelG, GetXandYSelGReal

variance_threshold = 0.9

def SamplingGenes(genenames, sample_sizes=[1000, 2000, 5000]):
    sampled_genes = {}
    for size in sample_sizes:
        sampled_genes[size] = np.random.choice(genenames, size=size, replace=False)
    return sampled_genes

if variance_threshold == 0.3:
    item = 'high_variance'
elif variance_threshold == 0.6:
    item = 'medium_variance'
elif variance_threshold == 0.9:
    item = 'low_variance'

genename_98 = var_genes_intv[item]
print(len(genename_98))
sampled_genes = SamplingGenes(genename_98)

GTE_x_r1, GTE_y_r1, celltypes = GetXandYSelG(simudata_GTE, sampled_genes[list(sampled_genes.keys())[0]], scaler="mms") 
GTE_x_r2, GTE_y_r2, celltypes = GetXandYSelG(simudata_GTE, sampled_genes[list(sampled_genes.keys())[1]], scaler="mms") 
GTE_x_r3, GTE_y_r3, celltypes = GetXandYSelG(simudata_GTE, sampled_genes[list(sampled_genes.keys())[2]], scaler="mms") 

HPA_x_r1, HPA_y_r1, celltypes = GetXandYSelG(simudata_HPA, sampled_genes[list(sampled_genes.keys())[0]], scaler="mms") 
HPA_x_r2, HPA_y_r2, celltypes = GetXandYSelG(simudata_HPA, sampled_genes[list(sampled_genes.keys())[1]], scaler="mms") 
HPA_x_r3, HPA_y_r3, celltypes = GetXandYSelG(simudata_HPA, sampled_genes[list(sampled_genes.keys())[2]], scaler="mms") 

print(GTE_x_r1.shape)
print(GTE_x_r2.shape)
print(GTE_x_r3.shape)

print(HPA_x_r1.shape)
print(HPA_x_r2.shape)
print(HPA_x_r3.shape)

# real数据
real_pth = "../result/expr/all_exo_tpm_filter.txt"
real_x_r1 = GetXandYSelGReal(real_pth, sampled_genes[list(sampled_genes.keys())[0]], scaler="mms") 
real_x_r2 = GetXandYSelGReal(real_pth, sampled_genes[list(sampled_genes.keys())[1]], scaler="mms") 
real_x_r3 = GetXandYSelGReal(real_pth, sampled_genes[list(sampled_genes.keys())[2]], scaler="mms") 

# print(real_x_r1.shape)
# print(real_x_r2.shape)
# print(real_x_r3.shape)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
5627
Scaling...
Using minmax scaler...
Scaling...
Using minmax scaler...
Scaling...
Using minmax scaler...
Scaling...
Using minmax scaler...
Scaling...
Using minmax scaler...
Scaling...
Using minmax scaler...
(5000, 1000)
(5000, 2000)
(5000, 5000)
(50, 1000)
(50, 2000)
(50, 5000)
Scaling...
Using minmax scaler...
Scaling...
Using minmax scaler...
Scaling...
Using minmax scaler...


# 拆分GTE仿真数据

In [69]:
%load_ext autoreload
%autoreload 2
from data_process import DataSplitTrValTe

GTE_x_r1_train, GTE_x_r1_val, GTE_x_r1_test, GTE_y_r1_train, GTE_y_r1_val, GTE_y_r1_test = DataSplitTrValTe(GTE_x_r1, GTE_y_r1)
GTE_x_r2_train, GTE_x_r2_val, GTE_x_r2_test, GTE_y_r2_train, GTE_y_r2_val, GTE_y_r2_test = DataSplitTrValTe(GTE_x_r2, GTE_y_r2)
GTE_x_r3_train, GTE_x_r3_val, GTE_x_r3_test, GTE_y_r3_train, GTE_y_r3_val, GTE_y_r3_test = DataSplitTrValTe(GTE_x_r3, GTE_y_r3)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Save data

In [70]:
data_to_save = {

    'simudata_GTE': simudata_GTE,
    'label_GTE': list(label_GTE),
    'simudata_HPA': simudata_HPA,
    'label_HPA': list(label_HPA),

    'genename_98': genename_98,
    'sampled_genes': sampled_genes,
    'celltypes': celltypes,

    'GTE_x_r1': GTE_x_r1,
    'GTE_y_r1': GTE_y_r1,
    'GTE_x_r2': GTE_x_r2,
    'GTE_y_r2': GTE_y_r2,
    'GTE_x_r3': GTE_x_r3,
    'GTE_y_r3': GTE_y_r3,

    'HPA_x_r1': HPA_x_r1,
    'HPA_y_r1': HPA_y_r1,
    'HPA_x_r2': HPA_x_r2,
    'HPA_y_r2': HPA_y_r2,
    'HPA_x_r3': HPA_x_r3,
    'HPA_y_r3': HPA_y_r3,

    'real_x_r1': real_x_r1,
    'real_x_r2': real_x_r2,
    'real_x_r3': real_x_r3,

    'GTE_x_r1_train': GTE_x_r1_train,
    'GTE_x_r1_val': GTE_x_r1_val,
    'GTE_x_r1_test': GTE_x_r1_test,
    'GTE_y_r1_train': GTE_y_r1_train,
    'GTE_y_r1_val': GTE_y_r1_val,
    'GTE_y_r1_test': GTE_y_r1_test,

    'GTE_x_r2_train': GTE_x_r2_train,
    'GTE_x_r2_val': GTE_x_r2_val,
    'GTE_x_r2_test': GTE_x_r2_test,
    'GTE_y_r2_train': GTE_y_r2_train,
    'GTE_y_r2_val': GTE_y_r2_val,
    'GTE_y_r2_test': GTE_y_r2_test,

    'GTE_x_r3_train': GTE_x_r3_train,
    'GTE_x_r3_val': GTE_x_r3_val,
    'GTE_x_r3_test': GTE_x_r3_test,
    'GTE_y_r3_train': GTE_y_r3_train,
    'GTE_y_r3_val': GTE_y_r3_val,
    'GTE_y_r3_test': GTE_y_r3_test
}

with open(f'../result/data_variFeats/Stim_data_{variance_threshold}.pkl', 'wb') as file:
    pickle.dump(data_to_save, file)